# All Town Hall transitions — days to max

Run the scheduler for every TH transition from TH8→TH9 up to TH17→TH18, with each preset (max / balanced / rush) and m ∈ {1, 2, 3, 6} builders.

Output: a reference table of "days to clear this transition" so a player can plan their progression. Also a stacked-bar chart showing cumulative time TH8→TH18.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data.loaders import load_troops_spells_heroes, load_buildings_xlsx, add_town_hall_gate
from src.data.schema import Track
from src.optim.selector import load_profile, apply_profile
from src.optim.lpt import lpt_schedule
from src.optim.cpsat import cpsat_schedule

JSON_PATH = ROOT / 'data_repo/clash-of-clans-data/output/troopUpgradeStats.json'
XLSX_PATH = ROOT / 'data_repo/structs_data.xlsx'
CONFIG_DIR = ROOT / 'config'

TH_RANGE = list(range(8, 18))  # TH8 -> TH9 through TH17 -> TH18
BUILDER_COUNTS = [1, 2, 3, 6]
PRESETS = ['max', 'balanced', 'rush']
print(f'Will compute {len(TH_RANGE)} transitions x {len(BUILDER_COUNTS)} builder counts x {len(PRESETS)} presets = {len(TH_RANGE)*len(BUILDER_COUNTS)*len(PRESETS)} runs.')

Will compute 10 transitions x 4 builder counts x 3 presets = 120 runs.


In [2]:
profiles = {p: load_profile(CONFIG_DIR / f'selection_{p}.yaml') for p in PRESETS}
rows = []

for start_th in TH_RANGE:
    target_th = start_th + 1
    try:
        troops = load_troops_spells_heroes(JSON_PATH, target_th=target_th, starting_th=start_th)
        bld = load_buildings_xlsx(XLSX_PATH, target_th=target_th, starting_th=start_th)
        jobs_all = add_town_hall_gate(troops + bld, target_th=target_th)
    except Exception as e:
        print(f'TH{start_th}->TH{target_th}: load error {e}')
        continue
    if not jobs_all:
        continue

    for preset in PRESETS:
        jobs = apply_profile(jobs_all, profiles[preset])
        for m in BUILDER_COUNTS:
            lpt = lpt_schedule(jobs, builders=m)
            cps = cpsat_schedule(jobs, builders=m, time_limit_sec=4, lpt_upper_bound=lpt.makespan_sec)
            rows.append({
                'transition': f'TH{start_th}->TH{target_th}',
                'preset': preset,
                'builders': m,
                'LPT_days': round(lpt.makespan_days, 1),
                'CPSAT_days': round(cps.schedule.makespan_days, 1),
            })

results = pd.DataFrame(rows)
print(f'Completed {len(results)} runs.')
results.head()

Completed 120 runs.


,transition,preset,builders,LPT_days,CPSAT_days
0,TH8->TH9,max,1,117.2,117.2
1,TH8->TH9,max,2,79.3,58.6
2,TH8->TH9,max,3,64.0,39.1
3,TH8->TH9,max,6,41.8,34.1
4,TH8->TH9,balanced,1,98.2,98.2


## Pivot — CP-SAT makespan, days, m=6 builders

In [3]:
m6 = results[results['builders']==6].pivot(index='transition', columns='preset', values='CPSAT_days')
m6 = m6[PRESETS]
m6

preset,max,balanced,rush
transition,,,
TH10->TH11,87.5,36.8,30.0
TH11->TH12,130.0,52.8,50.0
TH12->TH13,174.0,73.0,45.2
TH13->TH14,179.5,79.8,47.0
TH14->TH15,274.5,141.0,76.0
TH15->TH16,337.0,120.5,72.5
TH16->TH17,372.0,151.0,80.0
TH17->TH18,721.2,252.5,107.0
TH8->TH9,34.1,24.9,24.9


In [4]:
import plotly.express as px
long_df = results[results['builders']==6].copy()
long_df['transition_num'] = long_df['transition'].str.extract(r'TH(\d+)').astype(int)
long_df = long_df.sort_values('transition_num')
fig = px.bar(long_df, x='transition', y='CPSAT_days', color='preset',
             barmode='group', text='CPSAT_days',
             title='Days to clear each TH transition (CP-SAT, 6 builders)')
fig.update_traces(textposition='outside')
fig.show()

In [5]:
# Cumulative days to reach each TH from TH8
cum_rows = []
for preset in PRESETS:
    sub = long_df[long_df['preset']==preset].sort_values('transition_num')
    cum = 0
    for _, r in sub.iterrows():
        cum += r['CPSAT_days']
        cum_rows.append({
            'preset': preset,
            'reached_TH': int(r['transition'].split('->TH')[1]),
            'cumulative_days': cum,
            'cumulative_years': round(cum/365.25, 2),
        })
cum_df = pd.DataFrame(cum_rows)
fig = px.line(cum_df, x='reached_TH', y='cumulative_days', color='preset', markers=True,
              title='Cumulative days TH8 -> TH-X by preset (6 builders, CP-SAT)')
fig.update_yaxes(title='Cumulative days')
fig.show()
cum_df.pivot(index='reached_TH', columns='preset', values='cumulative_years')

preset,balanced,max,rush
reached_TH,,,
9,0.07,0.09,0.07
10,0.15,0.28,0.13
11,0.25,0.51,0.22
12,0.40,0.87,0.35
13,0.60,1.35,0.48
14,0.82,1.84,0.60
15,1.20,2.59,0.81
16,1.53,3.51,1.01
17,1.95,4.53,1.23


## Marginal value of each additional builder, by transition (max preset)

In [6]:
sweep = results[results['preset']=='max'].pivot(index='transition', columns='builders', values='CPSAT_days')
sweep['savings_1to2_d'] = sweep[1] - sweep[2]
sweep['savings_2to3_d'] = sweep[2] - sweep[3]
sweep['savings_3to6_d'] = sweep[3] - sweep[6]
sweep

builders,1,2,3,6,savings_1to2_d,savings_2to3_d,savings_3to6_d
transition,,,,,,,
TH10->TH11,248.1,124.0,87.5,87.5,124.1,36.5,0.0
TH11->TH12,354.0,177.0,130.0,130.0,177.0,47.0,0.0
TH12->TH13,480.7,240.3,174.0,174.0,240.4,66.3,0.0
TH13->TH14,523.5,261.8,179.5,179.5,261.7,82.3,0.0
TH14->TH15,698.8,349.5,274.5,274.5,349.3,75.0,0.0
TH15->TH16,757.0,378.5,337.0,337.0,378.5,41.5,0.0
TH16->TH17,943.5,472.0,372.0,372.0,471.5,100.0,0.0
TH17->TH18,1355.0,721.2,721.2,721.2,633.8,0.0,0.0
TH8->TH9,117.2,58.6,39.1,34.1,58.6,19.5,5.0


## Findings

Read the sweep table: at higher THs (13+), the marginal benefit of builders 3→6 is small or zero for the `max` preset because the Lab dominates. At lower THs (8-10), more builders help significantly because there's less lab work relative to builder work.

The cumulative days TH8→TH18 number is a real eye-opener: even with 6 builders and aggressive rushing, going from TH8 all the way to maxed TH18 takes years of in-game time.